In [1]:
import os
import sys
from pathlib import Path

repo_dir = Path("/content/SignLanguageGP")

if not repo_dir.exists():
    get_ipython().system("git clone https://github.com/AlphaTeam-YU/SignLanguageGP.git /content/SignLanguageGP")
else:
    get_ipython().system("git -C /content/SignLanguageGP pull")

os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

try:
    import mediapipe  # noqa: F401
except ModuleNotFoundError:
    get_ipython().system("pip install -q mediapipe")

print("Ready to import from SignLanguageGP")

Cloning into '/content/SignLanguageGP'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 148 (delta 63), reused 24 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 82.30 KiB | 458.00 KiB/s, done.
Resolving deltas: 100% (63/63), done.
Ready to import from SignLanguageGP


In [11]:
import importlib
import numpy as np
import words.extract as ext_mod
from words.normalize import WordNormalizer
import json
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
from words.extract import WordLandmarkExtractor
from words.normalize import WordNormalizer
import cv2
from sklearn.model_selection import train_test_split

In [2]:
importlib.reload(ext_mod)

video_path = "/content/drive/MyDrive/Sign Language Graduation Project/words/Justdata26-main/BABY/1.mp4"
ext = ext_mod.WordLandmarkExtractor()
try:
    seq = ext.process_video(video_path)
    print("Shape:", seq.shape)
    print("dtype:", seq.dtype)
    print("Has NaN/Inf:", bool(np.isnan(seq).any() or np.isinf(seq).any()))
    print("Min / Max:", float(seq.min()), "/", float(seq.max()))
finally:
    ext.close()

Shape: (50, 378)
dtype: float32
Has NaN/Inf: False
Min / Max: -0.525112509727478 / 1.055997610092163


In [4]:

norm = WordNormalizer()  # أو WordNormalizer("configs/words.yaml")
seq_norm = norm.normalize_sequence(seq)

print("Normalized shape:", seq_norm.shape)
print("Normalized dtype:", seq_norm.dtype)

Normalized shape: (50, 378)
Normalized dtype: float32


In [5]:
print("Has NaN/Inf after norm:", bool(np.isnan(seq_norm).any() or np.isinf(seq_norm).any()))
print("Norm min / max:", float(seq_norm.min()), "/", float(seq_norm.max()))

Has NaN/Inf after norm: False
Norm min / max: -1.4807072877883911 / 2.3346283435821533


In [7]:
DATA_DIR = Path("/content/drive/MyDrive/Sign Language Graduation Project/words/processed_data/unique_videos")  # مجلد يحتوي على مجلدات الكلمات (كل كلمة مجلد)
OUT_DIR = Path("/content/drive/MyDrive/Sign Language Graduation Project/words/processed_data/preprocessed_words")
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

OUT_DIR.mkdir(parents=True, exist_ok=True)

extractor = WordLandmarkExtractor(config_path="configs/words.yaml")
normalizer = WordNormalizer(config_path="configs/words.yaml")

# 3. قراءة أسماء الكلاسات (أسماء المجلدات) وترتيبها
class_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
label_to_idx = {d.name: idx for idx, d in enumerate(class_dirs)}

X_list = []
y_list = []
failed_videos = []

try:
    for class_dir in class_dirs:
        label_idx = label_to_idx[class_dir.name]
        videos = sorted([f for f in class_dir.iterdir() if f.suffix.lower() in VIDEO_EXTS])

        for video_path in tqdm(videos, desc=f"Class: {class_dir.name}"):
            try:
                raw_seq = extractor.process_video(str(video_path))       # (50, 378)
                clean_seq = normalizer.normalize_sequence(raw_seq)       # (50, 378)

                if np.isnan(clean_seq).any() or np.isinf(clean_seq).any():
                    failed_videos.append((str(video_path), "NaN/Inf detected"))
                    continue

                X_list.append(clean_seq)
                y_list.append(label_idx)
            except Exception as e:
                failed_videos.append((str(video_path), str(e)))
finally:
    extractor.close()

X_clean = np.asarray(X_list, dtype=np.float32)
y_labels = np.asarray(y_list, dtype=np.int64)

np.save(OUT_DIR / "X_clean.npy", X_clean)
np.save(OUT_DIR / "y_labels.npy", y_labels)
(OUT_DIR / "label_map.json").write_text(
    json.dumps(label_to_idx, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Done!")
print("X_clean shape:", X_clean.shape, "| dtype:", X_clean.dtype)
print("y_labels shape:", y_labels.shape, "| dtype:", y_labels.dtype)
print("Classes:", label_to_idx)
print("Failed count:", len(failed_videos))

Class: BABY:   0%|          | 0/6 [00:00<?, ?it/s]

Class: DAD:   0%|          | 0/7 [00:00<?, ?it/s]

Class: DIRECTIONS:   0%|          | 0/7 [00:00<?, ?it/s]

Class: DOCTOR:   0%|          | 0/7 [00:00<?, ?it/s]

Class: EARTH:   0%|          | 0/6 [00:00<?, ?it/s]

Class: EVENING:   0%|          | 0/5 [00:00<?, ?it/s]

... (13 outputs left out)

Class: WINTER:   0%|          | 0/6 [00:00<?, ?it/s]

Class: WORKER:   0%|          | 0/7 [00:00<?, ?it/s]

Class: WORKING:   0%|          | 0/5 [00:00<?, ?it/s]

Done!
X_clean shape: (133, 50, 378) | dtype: float32
y_labels shape: (133,) | dtype: int64
Classes: {'BABY': 0, 'DAD': 1, 'DIRECTIONS': 2, 'DOCTOR': 3, 'EARTH': 4, 'EVENING': 5, 'FAMILY': 6, 'FREIND': 7, 'GIRL': 8, 'GREEN': 9, 'HELLO': 10, 'ILL': 11, 'MAM': 12, 'ME': 13, 'MORNING': 14, 'TEACHER': 15, 'TODAY': 16, 'TWINS': 17, 'WEEK': 18, 'WINTER': 19, 'WORKER': 20, 'WORKING': 21}
Failed count: 1


In [8]:
out_dir = Path("/content/drive/MyDrive/Sign Language Graduation Project/words/processed_data/preprocessed_words")
failed_path = out_dir / "failed_videos.json"

if failed_path.exists():
    failed = json.loads(failed_path.read_text(encoding="utf-8"))
    for vid_path, reason in failed:
        print("Video:", vid_path)
        print("Reason:", reason)
        cap = cv2.VideoCapture(vid_path)
        print("Opened:", cap.isOpened(), "| Frame count:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
        cap.release()
else:
    print("No failed_videos.json found at:", failed_path)

No failed_videos.json found at: /content/drive/MyDrive/Sign Language Graduation Project/words/processed_data/preprocessed_words/failed_videos.json


In [10]:
out_dir = Path("/content/drive/MyDrive/Sign Language Graduation Project/words/processed_data/preprocessed_words")
out_dir.mkdir(parents=True, exist_ok=True)

X = np.asarray(X_clean, dtype=np.float32)
y = np.asarray(y_labels, dtype=np.int64)

np.save(out_dir / "X_processed.npy", X)
np.save(out_dir / "y_labels.npy", y)
(out_dir / "label_map.json").write_text(
    json.dumps(label_to_idx, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Saved successfully to:", out_dir)
print("X shape:", X.shape, "| y shape:", y.shape, "| classes:", len(label_to_idx))


try:
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
    )
except ValueError:
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=None
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42, stratify=None
    )

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

Saved successfully to: /content/drive/MyDrive/Sign Language Graduation Project/words/processed_data/preprocessed_words
X shape: (133, 50, 378) | y shape: (133,) | classes: 22
Train: (93, 50, 378) (93,)
Val:   (20, 50, 378) (20,)
Test:  (20, 50, 378) (20,)
